[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/04_shortest_paths_algorithms/first_principles.ipynb)

# Topic 04: Shortest Paths Algorithms

## 1. First-Principles Intuition & Motivation

Stand at a vertex $s$ of a weighted network and ask the simplest possible question: *what is the cheapest way to reach $v$?* Naively, the answer requires comparing all $s$–$v$ paths, and their number can grow exponentially with $n$. Yet every shortest-path algorithm runs in polynomial time. Why?

The reason is a single structural fact:

> If $P = s \to \dots \to u \to \dots \to v$ is a shortest $s$–$v$ path, then its prefix $s \to \dots \to u$ is a shortest $s$–$u$ path.

Otherwise we could splice in a cheaper prefix and beat $P$. This **optimal substructure** collapses the exponential search: instead of storing paths, we store one number per vertex — the current best known cost $d(v)$ — and improve it with a single local rule.

### The relaxation primitive

Every algorithm in this module is built from one operation. Given an edge $(u,v)$ with weight $w(u,v)$:

$$
\text{RELAX}(u,v): \quad \text{if } d(u) + w(u,v) \lt d(v) \text{ then } d(v) \leftarrow d(u) + w(u,v), \ \pi(v) \leftarrow u
$$

Two invariants make this safe and eventually exact:

- **Upper-bound invariant**: at all times $d(v)$ is the cost of some genuine $s$–$v$ walk (or $+\infty$), hence $d(v) \ge \delta(s,v)$, where $\delta$ denotes the true distance. Relaxation never produces a *fictitious* shortcut.
- **Path-relaxation invariant**: if the edges of a shortest $s$–$v$ path are relaxed *in order* (possibly with other relaxations interleaved), then afterwards $d(v) = \delta(s,v)$.

The algorithms differ only in the *schedule* used to guarantee the second invariant: BFS uses a FIFO queue (unit weights), Dijkstra a priority queue (nonnegative weights), Bellman–Ford brute-force repetition (arbitrary weights), Floyd–Warshall a dynamic program over intermediate vertices, and A$^{\ast}$ a heuristically reordered priority queue.

### Why weights change everything

Consider the three regimes:

1. **Unit weights.** Cost equals hop count. Exploring the graph in waves — all vertices at distance 1, then 2, then 3 — produces distances for free. This is breadth-first search.
2. **Nonnegative weights.** Waves no longer align with hops, but a *monotonicity* survives: extending a path never decreases its cost. So the globally smallest tentative label must already be final — you cannot reach it more cheaply by detouring through something even more expensive. This licenses greed, i.e. Dijkstra.
3. **Arbitrary weights.** A negative edge can make a detour cheaper than a direct route, destroying monotonicity. Nothing may be finalized early; instead we iterate the Bellman update until it stops changing, which takes $n-1$ rounds because shortest paths are simple.

If a *negative cycle* is reachable, no finite optimum exists: circulate the cycle forever and drive the cost to $-\infty$. The problem is then not hard, it is ill-posed — and detecting that is itself a valuable output.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition (weighted digraph, walk, path).** Let $G = (V, E)$ be a directed graph with $n = \vert V \vert$, $m = \vert E \vert$ and weights $w : E \to \mathbb{R}$. A **walk** is a sequence $v_0, v_1, \dots, v_k$ with $(v_{i-1}, v_i) \in E$; its weight is $w(P) = \sum_{i=1}^{k} w(v_{i-1}, v_i)$. A **path** is a walk with no repeated vertex.

**Definition (distance).** For $s, v \in V$,

$$
\delta(s,v) = \begin{cases} \min \lbrace w(P) : P \text{ an } s\text{–}v \text{ walk} \rbrace & \text{if such a walk exists and the min is attained} \\ +\infty & \text{if no } s\text{–}v \text{ walk exists} \\ -\infty & \text{if some } s\text{–}v \text{ walk meets a negative cycle} \end{cases}
$$

**Definition (shortest-path tree).** A subgraph $T$ rooted at $s$ containing, for each reachable $v$, a unique $s$–$v$ path of weight $\delta(s,v)$. The predecessor array $\pi$ maintained by relaxation encodes such a tree at termination.

**Definition (negative cycle).** A directed cycle $C$ with $w(C) \lt 0$.

**Definition (heuristic, admissible, consistent).** For a target $t$, a function $h : V \to \mathbb{R}_{\ge 0}$ is **admissible** if $h(v) \le \delta(v,t)$ for all $v$, and **consistent** (monotone) if $h(t) = 0$ and $h(u) \le w(u,v) + h(v)$ for every edge $(u,v)$.

### Theorem statements

**Theorem A (optimal substructure).** Every subpath of a shortest path is a shortest path between its endpoints.

**Theorem B (Bellman fixed point).** If no negative cycle is reachable from $s$, the distance vector $\delta(s, \cdot)$ is the unique solution of

$$
d(s) = 0, \qquad d(v) = \min_{(u,v) \in E} \lbrace d(u) + w(u,v) \rbrace \quad (v \neq s)
$$

that is attainable by walks; equivalently it is the largest vector satisfying $d(v) \le d(u) + w(u,v)$ for all edges, with $d(s) \le 0$.

**Theorem C (BFS correctness).** On a graph with unit weights, BFS from $s$ computes $d(v) = \delta(s,v)$ for all $v$ in $O(n + m)$ time, and vertices leave the queue in nondecreasing distance order.

**Theorem D (Dijkstra correctness).** If $w(e) \ge 0$ for all $e$, then when Dijkstra extracts a vertex $u$ from the priority queue, $d(u) = \delta(s,u)$; the algorithm therefore terminates with exact distances.

**Theorem E (Bellman–Ford).** After $k$ passes over all edges, $d(v) \le$ the minimum weight of any $s$–$v$ walk using at most $k$ edges. Consequently, if no negative cycle is reachable, $n-1$ passes suffice; and some edge still relaxes on the $n$-th pass **iff** a negative cycle is reachable from $s$.

**Theorem F (Floyd–Warshall).** Let $d^{(k)}(i,j)$ be the minimum weight of an $i$–$j$ path whose interior vertices lie in $\lbrace 1, \dots, k \rbrace$. Then $d^{(k)}(i,j) = \min \lbrace d^{(k-1)}(i,j),\ d^{(k-1)}(i,k) + d^{(k-1)}(k,j) \rbrace$, and $d^{(n)} = \delta$ provided no negative cycle exists.

**Theorem G (A$^{\ast}$).** With a consistent heuristic $h$, A$^{\ast}$ using key $f(v) = d(v) + h(v)$ is exactly Dijkstra on the reduced weights $w_h(u,v) = w(u,v) + h(v) - h(u) \ge 0$, hence returns an optimal $s$–$t$ path and never expands a vertex twice.

**Theorem H (DAG shortest paths).** In a directed acyclic graph, relaxing all outgoing edges once per vertex in topological order computes all distances in $O(n + m)$, for arbitrary (possibly negative) weights.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: Optimal substructure and the Bellman equation

**Theorem A.** Let $P = \langle v_0 = s, v_1, \dots, v_k = v \rangle$ be a shortest $s$–$v$ path and let $0 \le i \le j \le k$. Then the subpath $P_{ij} = \langle v_i, \dots, v_j \rangle$ is a shortest $v_i$–$v_j$ path.

**Proof.** Decompose $w(P) = w(P_{0i}) + w(P_{ij}) + w(P_{jk})$. If some $v_i$–$v_j$ path $Q$ had $w(Q) \lt w(P_{ij})$, then the walk $P_{0i} \cdot Q \cdot P_{jk}$ would be an $s$–$v$ walk of weight $w(P) - w(P_{ij}) + w(Q) \lt w(P)$. Since no negative cycle is reachable, the minimum over walks equals the minimum over paths (any repeated vertex can be excised, removing a cycle of nonnegative weight), so this contradicts the optimality of $P$. $\blacksquare$

**Consequence (Theorem B).** Let $v \neq s$ be reachable and let $P$ be a shortest $s$–$v$ path with last edge $(u,v)$. By Theorem A, $w(P_{su}) = \delta(s,u)$, so $\delta(s,v) = \delta(s,u) + w(u,v) \ge \min_{(x,v) \in E} \lbrace \delta(s,x) + w(x,v) \rbrace$. Conversely each candidate $\delta(s,x) + w(x,v)$ is the weight of a genuine $s$–$v$ walk, hence $\ge \delta(s,v)$. Both inequalities give

$$
\boxed{\delta(s,v) = \min_{(u,v) \in E} \lbrace \delta(s,u) + w(u,v) \rbrace, \qquad \delta(s,s) = 0}
$$

$\blacksquare$

### Proof 2: BFS computes unweighted distances

**Theorem C.** With $w \equiv 1$, BFS from $s$ sets $d(v) = \delta(s,v)$ for every $v$, and dequeues vertices in nondecreasing $d$.

**Proof (induction on distance level).** BFS maintains a FIFO queue; a vertex is *discovered* when first assigned a finite $d$ and enqueued.

*Claim 1 (upper bound).* $d(v) \ge \delta(s,v)$ always, since $d(v) = d(u) + 1$ for the discovering edge $(u,v)$ and induction gives a real walk of that length.

*Claim 2 (queue monotonicity).* If the queue holds $\langle u_1, \dots, u_r \rangle$ then $d(u_1) \le d(u_2) \le \dots \le d(u_r) \le d(u_1) + 1$. This holds initially ($\langle s \rangle$) and is preserved: dequeuing $u_1$ leaves the chain intact, and every vertex enqueued while processing $u_1$ receives label $d(u_1) + 1 \le d(u_r) + 1$.

*Claim 3 (exactness).* Suppose for contradiction that some vertex has $d(v) \gt \delta(s,v)$; choose one with minimum $\delta(s,v) = k \ge 1$. Let $u$ be its predecessor on a shortest path, so $\delta(s,u) = k-1$ and by minimality $d(u) = k-1$. When $u$ is dequeued, the edge $(u,v)$ is examined: either $v$ is still undiscovered — then $d(v) \leftarrow k$, contradiction — or $v$ was already discovered, and by Claim 2 it was labelled no later than $u$ was, so $d(v) \le d(u) + 1 = k$, again a contradiction.

Hence $d = \delta$ everywhere. Each vertex is enqueued once and each edge scanned once (twice if undirected), giving $O(n + m)$. $\blacksquare$

$$
\boxed{\text{BFS: unit-weight distances in } O(n + m)}
$$

### Proof 3: Dijkstra's correctness (induction on extraction order)

**Theorem D.** Assume $w(e) \ge 0$ for all edges. Let $S$ be the set of already-extracted (settled) vertices. Then the invariant

$$
d(u) = \delta(s,u) \quad \text{for every } u \in S
$$

holds throughout, so at termination all labels are exact.

**Proof (induction on $\vert S \vert$).**

*Base.* $S = \emptyset$ is vacuous; the first extraction is $s$ with $d(s) = 0 = \delta(s,s)$ (no negative weights exist to make it smaller).

*Step.* Suppose the invariant holds for $S$ and let $u \notin S$ be the vertex of minimum key extracted next. We must show $d(u) = \delta(s,u)$. Since $d(u) \ge \delta(s,u)$ always (upper-bound invariant), it suffices to prove $d(u) \le \delta(s,u)$. Assume $u$ is reachable, and let $P$ be a shortest $s$–$u$ path. Because $s \in S$ and $u \notin S$, $P$ has a first vertex $y \notin S$; let $x \in S$ be its predecessor on $P$ (possibly $x = s$, possibly $y = u$).

By the induction hypothesis $d(x) = \delta(s,x)$, and edge $(x,y)$ was relaxed when $x$ was extracted, hence

$$
d(y) \le d(x) + w(x,y) = \delta(s,x) + w(x,y) = \delta(s,y)
$$

the last equality by optimal substructure applied to the prefix of $P$ ending at $y$. Combining with the upper-bound invariant, $d(y) = \delta(s,y)$.

Now use **nonnegativity**: $y$ lies on $P$ before $u$, and the remaining segment $P_{yu}$ has weight $\ge 0$, so $\delta(s,y) \le \delta(s,u)$. Finally $u$ was chosen as the minimum-key unsettled vertex and $y$ is unsettled, so $d(u) \le d(y)$. Chaining,

$$
d(u) \le d(y) = \delta(s,y) \le \delta(s,u) \le d(u)
$$

forcing equality throughout. $\blacksquare$

$$
\boxed{\text{Dijkstra is correct exactly because } w \ge 0 \Rightarrow \delta(s,y) \le \delta(s,u) \text{ for } y \text{ on a shortest path to } u}
$$

**Where negativity breaks it.** The single step $\delta(s,y) \le \delta(s,u)$ is the only use of $w \ge 0$. With a negative edge later on $P$, a vertex $y$ with a *larger* label can still lie on the optimal route to $u$, and extracting $u$ first finalizes a wrong value. Counterexample: $s \to a$ with weight $1$, $s \to b$ with weight $4$, $b \to a$ with weight $-3$. Dijkstra settles $a$ at $1$, while $\delta(s,a) = 4 - 3 = 1$ — equal here, but changing $s \to b$ to $2$ gives $\delta(s,a) = -1$ while Dijkstra still reports $1$.

### Proof 4: Bellman–Ford — the hop-count bound and negative-cycle detection

**Theorem E (part 1).** Let $d_k(v)$ denote the label after $k$ full passes over $E$ (each pass relaxes every edge once, in any fixed order). Then

$$
d_k(v) \le \delta_k(s,v) := \min \lbrace w(W) : W \text{ an } s\text{–}v \text{ walk with at most } k \text{ edges} \rbrace
$$

**Proof (induction on $k$).** For $k = 0$: $d_0(s) = 0 = \delta_0(s,s)$ and $d_0(v) = +\infty = \delta_0(s,v)$ for $v \neq s$. Assume the claim after $k-1$ passes. Fix $v$ and let $W$ be an optimal $s$–$v$ walk with at most $k$ edges; if it has $0$ edges then $v = s$ and $d_k(s) \le 0$. Otherwise write $W = W' \cdot (u,v)$ where $W'$ has at most $k-1$ edges, so $d_{k-1}(u) \le w(W')$ by hypothesis. During pass $k$ the edge $(u,v)$ is relaxed, at which moment the current label of $u$ is $\le d_{k-1}(u)$ (labels never increase), so afterwards

$$
d_k(v) \le d_{k-1}(u) + w(u,v) \le w(W') + w(u,v) = w(W)
$$

$\blacksquare$

**Corollary.** If no negative cycle is reachable, every shortest walk can be taken simple, hence has at most $n-1$ edges, and $d_{n-1}(v) \le \delta(s,v)$. With the upper-bound invariant $d \ge \delta$ this gives $d_{n-1} = \delta$: **$n-1$ passes suffice**, at cost $O(nm)$.

$$
\boxed{d_{n-1}(v) = \delta(s,v) \quad \text{when no negative cycle is reachable}}
$$

### Proof 4b: The $n$-th pass detects negative cycles

**Theorem E (part 2).** After $n-1$ passes, some edge $(u,v)$ still satisfies $d(u) + w(u,v) \lt d(v)$ **iff** a negative cycle is reachable from $s$.

**Proof.**

($\Leftarrow$) Suppose the reachable cycle $C = \langle v_0, v_1, \dots, v_r = v_0 \rangle$ has $w(C) \lt 0$, and suppose for contradiction that no edge relaxes further, i.e. $d(v_{i-1}) + w(v_{i-1}, v_i) \ge d(v_i)$ for all $i$. All these labels are finite (the cycle is reachable). Summing over $i = 1, \dots, r$:

$$
\sum_{i=1}^{r} d(v_{i-1}) + \sum_{i=1}^{r} w(v_{i-1}, v_i) \ge \sum_{i=1}^{r} d(v_i)
$$

The two label sums are identical (the cycle visits the same vertex set), so they cancel, leaving $w(C) \ge 0$ — contradicting $w(C) \lt 0$.

($\Rightarrow$) If no negative cycle is reachable, then $d_{n-1} = \delta$ by the corollary above, and $\delta$ satisfies $\delta(v) \le \delta(u) + w(u,v)$ for every edge (Theorem B), so no relaxation succeeds. $\blacksquare$

$$
\boxed{\text{extra relaxation on pass } n \iff \text{reachable negative cycle}}
$$

**Extracting the cycle.** If edge $(u,v)$ relaxes on pass $n$, follow predecessors $\pi$ back from $v$ for $n$ steps; the vertex reached certainly lies on a negative cycle, and tracing $\pi$ from it until repetition prints the cycle.

### Proof 5: Floyd–Warshall by dynamic programming over intermediate vertices

**Theorem F.** Number the vertices $1, \dots, n$ and define $d^{(k)}(i,j)$ as the minimum weight of an $i$–$j$ path all of whose *interior* vertices belong to $\lbrace 1, \dots, k \rbrace$ (with $d^{(0)}(i,j) = w(i,j)$, $+\infty$ if absent, $0$ if $i = j$). Then

$$
d^{(k)}(i,j) = \min \lbrace d^{(k-1)}(i,j), \ d^{(k-1)}(i,k) + d^{(k-1)}(k,j) \rbrace
$$

and, absent negative cycles, $d^{(n)}(i,j) = \delta(i,j)$.

**Proof.** Let $P$ be an optimal $i$–$j$ path with interior vertices in $\lbrace 1, \dots, k \rbrace$; since there is no negative cycle we may take $P$ simple, so it visits $k$ at most once.

*Case 1: $k \notin P$.* Then all interior vertices lie in $\lbrace 1, \dots, k-1 \rbrace$ and $w(P) = d^{(k-1)}(i,j)$.

*Case 2: $k \in P$.* Split $P = P_1 \cdot P_2$ at $k$. Since $P$ is simple, neither piece uses $k$ internally, and by optimal substructure each is optimal for its endpoints with interior vertices in $\lbrace 1, \dots, k-1 \rbrace$. Hence $w(P) = d^{(k-1)}(i,k) + d^{(k-1)}(k,j)$.

The recurrence takes the better of the two cases, and both candidates are achievable, so it is exact. Taking $k = n$ removes all restrictions. $\blacksquare$

**Implementation note.** The update can be done **in place** on a single $n \times n$ matrix: when processing $k$, the entries $d(i,k)$ and $d(k,j)$ are unchanged by row/column $k$ updates (because $d(k,k) = 0$ in the absence of negative cycles), so no extra copy is needed. Total cost:

$$
\boxed{\Theta(n^3) \text{ time}, \ \Theta(n^2) \text{ space}; \quad \text{negative cycle} \iff d^{(n)}(i,i) \lt 0 \text{ for some } i}
$$

### Proof 6: A$^{\ast}$ is Dijkstra on reduced costs

**Theorem G.** Let $h$ be consistent: $h(t) = 0$ and $h(u) \le w(u,v) + h(v)$ for every edge. Define the **reduced cost**

$$
w_h(u,v) = w(u,v) + h(v) - h(u)
$$

Then (i) $w_h \ge 0$; (ii) for every $s$–$t$ path $P$, $w_h(P) = w(P) + h(t) - h(s) = w(P) - h(s)$; (iii) A$^{\ast}$ with key $f(v) = d(v) + h(v)$ expands vertices exactly as Dijkstra run with weights $w_h$, hence returns a shortest $s$–$t$ path and never re-expands a closed vertex.

**Proof.**

(i) Consistency is literally the inequality $w(u,v) + h(v) - h(u) \ge 0$.

(ii) The transformation telescopes: summing $w_h$ along $P = \langle s = v_0, \dots, v_k = t \rangle$,

$$
\sum_{i=1}^{k} \big( w(v_{i-1},v_i) + h(v_i) - h(v_{i-1}) \big) = w(P) + h(v_k) - h(v_0) = w(P) - h(s)
$$

Since $h(s)$ is a constant independent of $P$, the *ordering* of $s$–$t$ paths by $w_h$ is identical to their ordering by $w$: minimizers coincide.

(iii) If $d_h$ denotes Dijkstra's label under $w_h$, induction on relaxations gives $d_h(v) = d(v) + h(v) - h(s)$, so the priority key $d_h(v)$ differs from A$^{\ast}$'s $f(v) = d(v) + h(v)$ by the constant $h(s)$: the two algorithms extract vertices in the same order. By Theorem D applied to the nonnegative weights $w_h$, the labels are exact and no vertex needs reopening. Undoing the shift returns true distances. $\blacksquare$

$$
\boxed{\text{consistent } h \iff w_h \ge 0 \iff \text{A}^{\ast} \text{ is a correct Dijkstra run}}
$$

**Corollaries.**
- $h \equiv 0$ is consistent, so Dijkstra is the special case A$^{\ast}$ with no guidance.
- Consistency implies admissibility: applying $h(u) \le w(u,v) + h(v)$ along a shortest $u$–$t$ path telescopes to $h(u) \le \delta(u,t)$.
- Admissible-but-inconsistent heuristics still give optimal answers *if* closed vertices may be reopened, at the price of possibly exponential re-expansions.
- The same reweighting underlies **Johnson's algorithm**, which uses *from-source* potentials $p$ instead of *to-target* estimates: run Bellman–Ford from an artificial source to obtain $p$ with $p(v) \le p(u) + w(u,v)$, set $\hat{w}(u,v) = w(u,v) + p(u) - p(v) \ge 0$, then run $n$ Dijkstras for all-pairs distances in $O(nm + n^2 \log n)$ — better than $\Theta(n^3)$ on sparse graphs. Both are instances of the single rule "shift by a potential difference; only telescoping shifts preserve shortest paths."

### Proof 7: DAG shortest paths in one topological sweep

**Theorem H.** If $G$ is acyclic, relaxing every vertex's outgoing edges once, in topological order, yields $d = \delta$ in $O(n + m)$ — even with negative weights.

**Proof.** Let $v$ be reachable and let $P = \langle s = v_0, \dots, v_k = v \rangle$ be a shortest path. In a topological order, an edge always points forward, so $v_0 \prec v_1 \prec \dots \prec v_k$. The sweep therefore relaxes $(v_0,v_1)$, then later $(v_1,v_2)$, and so on — the edges of $P$ are relaxed **in order**. By the path-relaxation invariant, $d(v) = \delta(s,v)$ at the end. Topological sorting costs $O(n+m)$ and each edge is relaxed once. $\blacksquare$

Acyclicity also guarantees no negative cycles, so the problem is always well posed.

$$
\boxed{\text{DAG: } O(n + m) \text{ for arbitrary real weights}}
$$

**Longest paths for free.** Negating all weights turns shortest into longest, so critical-path analysis in project scheduling (PERT/CPM) and the longest-path layering used in DAG-structured neural architectures are the same sweep with a sign flip. In a general graph, by contrast, longest path is NP-hard — precisely because negation creates negative cycles.

## 4. Computational & Algorithmic Insights

### Algorithm comparison

| Algorithm | Weight assumption | Data structure | Time | Space |
|---|---|---|---|---|
| BFS | unit / uniform | FIFO queue | $O(n + m)$ | $O(n)$ |
| DAG sweep | any real, acyclic $G$ | topological order | $O(n + m)$ | $O(n)$ |
| Dijkstra (array) | $w \ge 0$ | unsorted array | $O(n^2 + m)$ | $O(n)$ |
| Dijkstra (binary heap) | $w \ge 0$ | binary min-heap | $O((n + m)\log n)$ | $O(n)$ |
| Dijkstra (Fibonacci heap) | $w \ge 0$ | Fibonacci heap | $O(m + n \log n)$ | $O(n)$ |
| Bellman–Ford | any real | edge list | $O(nm)$ | $O(n)$ |
| SPFA (queue-based BF) | any real | FIFO queue | $O(nm)$ worst, fast in practice | $O(n)$ |
| Floyd–Warshall | any real, no negative cycle | $n \times n$ matrix | $\Theta(n^3)$ | $\Theta(n^2)$ |
| Johnson | any real, no negative cycle | BF + $n$ Dijkstras | $O(nm + n^2 \log n)$ | $O(n^2)$ |
| A$^{\ast}$ | $w \ge 0$, consistent $h$ | priority queue | $O((n + m)\log n)$ worst; far less in practice | $O(n)$ |

### Where the binary-heap bound comes from

Dijkstra performs at most $n$ EXTRACT-MIN operations (one per vertex) and at most $m$ DECREASE-KEY operations (one per relaxed edge). With a binary heap both cost $O(\log n)$:

$$
T = n \cdot O(\log n) + m \cdot O(\log n) = O((n + m)\log n)
$$

A Fibonacci heap makes DECREASE-KEY $O(1)$ amortized while keeping EXTRACT-MIN at $O(\log n)$ amortized, giving $O(m + n\log n)$ — an improvement whenever $m = \omega(n)$. The "lazy" implementation used in practice skips DECREASE-KEY entirely: push a duplicate entry on every successful relaxation and discard stale pops. The heap then holds $O(m)$ entries and the bound becomes $O(m \log m) = O(m \log n)$, with much smaller constants.

**Special-weight speedups.**

| Structure | Technique | Time |
|---|---|---|
| Weights in $\lbrace 0, 1 \rbrace$ | 0–1 BFS with a deque (push-front for weight 0) | $O(n + m)$ |
| Small integer weights bounded by $C$ | Dial's buckets (monotone priority queue) | $O(m + nC)$ |
| Integer weights, word RAM | Thorup's algorithm | $O(m)$ for undirected graphs |
| Repeated queries, static graph | contraction hierarchies / hub labels | near-constant query after preprocessing |

### Pitfalls, invariants and verification strategy

- **Verify with the dual certificate.** A distance vector $d$ is correct iff (a) $d(v) \le d(u) + w(u,v)$ for every edge (*feasibility*, checkable in $O(m)$) and (b) every $v$ has a tight incoming edge, i.e. $d(v) = d(\pi(v)) + w(\pi(v), v)$ (*attainment*). This is a linear-programming complementary-slackness check and is the cheapest way to test any implementation.
- **Predecessor graph is a tree.** With exact labels the predecessor pointers contain no cycle: each $\pi$-step strictly decreases $d$ when weights are positive, and in general a $\pi$-cycle would have negative total weight.
- **Ties matter for reproducibility.** Multiple shortest paths mean multiple valid trees; deterministic tie-breaking (by vertex index) makes outputs comparable across implementations.
- **Floating-point weights.** Accumulated rounding can make relaxation oscillate; compare with a tolerance $d(u) + w \lt d(v) - \varepsilon$, or scale to integers.
- **Do not mix objectives.** Minimizing hop count and minimizing weight are different problems; if both matter, use lexicographic keys $(w, \text{hops})$ or perturb weights by $\varepsilon \cdot \text{hops}$.
- **Sanity checks.** $\delta$ obeys the triangle inequality $\delta(s,v) \le \delta(s,u) + \delta(u,v)$; distances from $s$ in an undirected graph must satisfy $\vert \delta(s,u) - \delta(s,v) \vert \le w(u,v)$ for every edge — a cheap randomized audit.

## 5. Real-World Physics & AI/ML Applications

### Physics, geometry and infrastructure

- **Geodesics and Fermat's principle.** Light takes the path of least optical length $\int n \, ds$; discretizing the medium into a grid turns this into a shortest-path problem, and fast-marching methods are literally Dijkstra on the Eikonal equation $\Vert \nabla T \Vert = 1/c(x)$.
- **Routing protocols.** OSPF runs Dijkstra on link-state maps; RIP and the original ARPANET protocol run distributed Bellman–Ford, where each router iterates the Bellman update using neighbours' tables — with the "count-to-infinity" pathology being exactly slow convergence of the fixed point.
- **Transport networks and navigation.** Road routing uses A$^{\ast}$ with the great-circle heuristic (admissible because straight-line distance never overestimates travel distance when speeds are bounded), plus contraction hierarchies for continent-scale queries.
- **Percolation and disordered media.** Minimum-energy paths in random landscapes (directed polymers, first-passage percolation) are shortest paths with random weights; their fluctuation exponents are a topic of statistical physics.
- **Seam carving and image segmentation.** Content-aware image resizing removes the minimum-energy vertical seam — a DAG shortest path over the pixel grid.

### AI and machine learning

- **Value iteration is Bellman–Ford.** For a deterministic MDP with rewards $-w$, the optimality equation $V(s) = \max_a \lbrace r(s,a) + V(s') \rbrace$ is the Bellman shortest-path equation with a sign flip; synchronous value iteration is exactly a full relaxation pass, and the discount factor $\gamma \lt 1$ is what guarantees the contraction that replaces the "no negative cycle" hypothesis. Prioritized sweeping is Dijkstra's priority queue transplanted into reinforcement learning.
- **Viterbi decoding.** Finding the most probable state sequence in an HMM or CRF is a shortest path in a trellis DAG with weights $-\log p$; the DAG sweep of Theorem H *is* the Viterbi algorithm, and beam search is its heuristically pruned A$^{\ast}$ cousin.
- **Search in planning and games.** A$^{\ast}$, weighted A$^{\ast}$ and its anytime variants dominate classical planning; learned heuristics (neural admissible bounds, pattern databases) are the modern route to shrinking the explored set while retaining the guarantees of Theorem G.
- **Graph embeddings and distance approximation.** Node2vec, Isomap and hyperbolic embeddings all try to encode $\delta(u,v)$ into a geometry where distance is cheap to evaluate; landmark/hub labelling gives $\vert \hat{\delta} - \delta \vert$ bounds via the triangle inequality $\vert \delta(s,u) - \delta(s,v) \vert \le \delta(u,v) \le \delta(u,\ell) + \delta(\ell,v)$.
- **Graph neural networks.** A GNN with $k$ message-passing layers can compute only $k$-hop information; simulating distances therefore needs depth comparable to the graph diameter, which is one precise statement of the over-squashing problem. Explicit distance encodings (shortest-path bias in Graphormer) inject Topic 04 quantities as features instead.
- **Word-level alignment and dynamic time warping.** Edit distance, DTW and CTC decoding are DAG shortest paths on an alignment lattice — same recurrence, different lattice.

### A worked micro-example: routing versus connecting

Take the triangle $V = \lbrace s, a, b \rbrace$ with $w(s,a) = 1$, $w(s,b) = 1.9$, $w(a,b) = 1$.

- **MST** (Topic 03): edges $sa$ and $ab$, total weight $2$. Cheapest way to *connect* everything.
- **Shortest-path tree from $s$**: edges $sa$ (cost 1) and $sb$ (cost 1.9), since routing $s \to a \to b$ costs $2 \gt 1.9$. Total tree weight $2.9$.

The two trees differ, and each is optimal for its own objective. In network design this is the latency-versus-capex trade-off; in machine learning it is the difference between building a sparse graph that preserves connectivity and one that preserves distances.

$$
\boxed{w(\mathrm{MST}) = 2 \lt w(\mathrm{SPT}) = 2.9, \quad \text{yet } \delta_{\mathrm{SPT}}(s,b) = 1.9 \lt \delta_{\mathrm{MST}}(s,b) = 2}
$$

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| Optimal substructure, relaxation invariants | Cormen–Leiserson–Rivest–Stein (4th ed.), Ch. 22 |
| Dijkstra's algorithm and correctness proof | Dijkstra (1959), *Numer. Math.* 1; CLRS §22.3 |
| Bellman–Ford, negative-cycle detection | Bellman (1958); Ford & Fulkerson (1962); CLRS §22.1 |
| Floyd–Warshall dynamic program | Floyd (1962), *CACM* 5(6); Warshall (1962); CLRS §23.2 |
| Johnson reweighting for sparse all-pairs | Johnson (1977), *J. ACM* 24(1); CLRS §23.3 |
| A$^{\ast}$, admissibility and consistency | Hart, Nilsson & Raphael (1968); Pearl, *Heuristics* (1984), Ch. 2–3 |
| Label-setting vs. label-correcting taxonomy | Ahuja, Magnanti & Orlin, *Network Flows*, Ch. 4–5 |
| LP duality view of shortest paths | Schrijver, *Combinatorial Optimization*, Ch. 7 |
| Priority queues and the $O(m + n\log n)$ bound | Fredman & Tarjan (1987), *J. ACM* 34(3) |
| Linear-time undirected shortest paths | Thorup (1999), *J. ACM* 46(3) |
| Value iteration and prioritized sweeping | Sutton & Barto (2018), Ch. 4; Moore & Atkeson (1993) |
| Fast marching / Eikonal solvers | Sethian (1996), *PNAS* 93(4) |
| Distances in graph representation learning | Hamilton (2020), *Graph Representation Learning*, Ch. 2–3 |

**Cross-links within this repository**

- Traversal foundations (BFS/DFS, connectivity): [`../02_traversal_and_connectivity/README.md`](../02_traversal_and_connectivity/README.md)
- MST contrast (cheapest connector vs. cheapest router): [`../03_trees_and_minimum_spanning_trees/README.md`](../03_trees_and_minimum_spanning_trees/README.md)
- Shortest augmenting paths inside flow algorithms: [`../05_flows_matchings_and_bipartite_graphs/README.md`](../05_flows_matchings_and_bipartite_graphs/README.md)
- Runnable Dijkstra/BFS demos: [`../computation.ipynb`](../computation.ipynb)
- Solved problem set for this topic: [`exercises.ipynb`](exercises.ipynb)